# Stochastic Goose — benchmark sweep (Colab, OFFLINE, env-step-capped, PARALLEL, Drive-persisted)

Runs Stochastic Goose offline (no API key) on all 4 games × N seeds.
- **Env-step cutoff** — each level gets a fixed **per-level env-step budget** (`PER_LEVEL_BUDGET`), the *same* cutoff rule we use to benchmark our own ICM/RND/leaky-RND methods. Wall-clock capping is **disabled** (`MAX_MINUTES` set huge) so the env-step budget is the *only* cutoff.
- **Parallel** — up to `MAX_PARALLEL` jobs run at once on one GPU. A single goose run is batch-1 GPU-latency-bound + CPU env-stepping, so it under-uses an A100; overlapping several recovers that idle time. Each job is its own OS process, so one crashing never takes the others down.
- **Local-first logging** — logs + results go to `/content/` first (never fails on Drive disconnect).
- **Incremental saves** — the runner overwrites its result JSON on every level-up / every ~2k actions / every ~2 min, so a disconnect loses almost nothing.
- **Drive backup** after every finished job *and* periodically mid-run, with auto-remount on disconnect.
- **No skip** — every `(game, seed)` is always (re)run from scratch; nothing is skipped. The runner overwrites its own result files, so re-running is safe.

> **Set Runtime ▸ GPU.** Goose's CNN is the bottleneck; GPU speeds it ~5×.

## 1. Setup (clone + install)

In [ ]:
import os, shutil
REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"; REPO = "/content/ProjectArceus"
%cd /content
if os.path.isdir(REPO):
    shutil.rmtree(REPO)              # ALWAYS start clean — avoids the stale-shallow-pull bug
!git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3" torch
import torch; print("CUDA:", torch.cuda.is_available())
# sanity: the runner MUST know the new per-level flags, else g50t/re86 crash on argparse
_chk = "replication/card_stochastic_goose/goose_offline_run.py"
assert "max-actions-per-level" in open(_chk).read(), "STALE runner — re-clone did not update it!"
print("runner OK (supports --max-actions-per-level / --torch-threads)")


## 2. Mount Drive + config

**Root cause of prior crash**: writing log files directly to `/content/drive/...` fails with
`OSError: [Errno 107] Transport endpoint is not connected` when the Drive FUSE mount disconnects
(common after ~1–2 h of inactivity). The whole `ThreadPoolExecutor` crashed.

**Fix**: all writes go to **local `/content/goose_*/`** first. Drive is only touched in
`try_drive_backup()` which catches all errors and auto-remounts.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path
from datetime import datetime

# ── local paths (writes here ALWAYS; Drive is backup) ────────────────────────
LOCAL_OUT = Path("/content/goose_results")   # runner writes JSON here
LOG_DIR   = Path("/content/goose_logs")      # stdout/stderr per job
LOCAL_OUT.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# ── Drive (optional backup) ──────────────────────────────────────────────────
DRIVE_OUT = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = Path("/content/drive/MyDrive/goose_results")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted → {DRIVE_OUT}")
    # Copy any results already on Drive to local (for the aggregation cell only; runs are NOT skipped)
    for f in DRIVE_OUT.glob("goose_*.json"):
        dst = LOCAL_OUT / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
            print(f"  pulled {f.name} from Drive")
except Exception as e:
    print(f"Drive unavailable ({e}) — saving locally only")

# ── sweep config ─────────────────────────────────────────────────────────────
RUNNER = "/content/ProjectArceus/replication/card_stochastic_goose/goose_offline_run.py"
GAMES  = ["ls20", "tu93", "re86", "g50t"]
SEEDS  = [0, 1]
NLEV   = 3

# Per-level ENV-STEP budget — the cutoff. Goose resets its model+buffer on every level advance,
# so each level gets a fresh budget exactly like our per-level ICM/RND/leaky-RND runs. These match
# the per-level caps used for our own methods in exp_014 (baseline icm/rnd column).
PER_LEVEL_BUDGET = {"ls20": 600_000, "tu93": 600_000, "re86": 1_000_000, "g50t": 600_000}
MAX_ACTIONS = {g: PER_LEVEL_BUDGET[g] * NLEV for g in GAMES}   # hard TOTAL ceiling = budget × levels
MAX_MINUTES = 100_000      # wall-clock cap DISABLED — per-level env-step budget is the SOLE cutoff
                           # (unconstrained run; Colab's own session limit is the only real wall-clock bound,
                           #  and incremental saves + Drive backup make a disconnect lose almost nothing)

# Parallelism — several goose runs at once on one GPU (each under-uses an A100 on batch-1 inference).
MAX_PARALLEL  = 4          # concurrent jobs; ~1–1.5 GB CUDA ctx each → 4 fits comfortably on A100 (40 GB)
TORCH_THREADS = 2          # cap CPU threads per process so parallel jobs don't oversubscribe cores
BACKUP_SECS   = 300        # also back up in-progress (incremental) results to Drive this often

print(f"LOCAL results → {LOCAL_OUT}")
print(f"LOCAL logs    → {LOG_DIR}")
print(f"Drive backup  → {DRIVE_OUT or 'disabled'}")
print(f"Per-level budget: {PER_LEVEL_BUDGET}  (total ceiling = ×{NLEV})")
print(f"Jobs: {len(GAMES)*len(SEEDS)} total ({GAMES} × seeds {SEEDS}), up to {MAX_PARALLEL} in parallel")

## 3. Run the sweep (parallel, env-step-capped, disconnect-safe)

- Up to `MAX_PARALLEL` jobs run concurrently, each as its own OS process (`subprocess.Popen`) — no shared `ThreadPoolExecutor`, so one job dying can't take the rest down (that was the prior crash).
- Each level is cut off at its `PER_LEVEL_BUDGET` env-steps inside the runner; `--max-actions` is a hard total ceiling and `--max-minutes` a wall-clock safety net.
- Logs go to **local** `/content/goose_logs/log_<game>_seed<s>.txt`. Results stream to **local** `/content/goose_results/` and are backed up to Drive after every finished job *and* every `BACKUP_SECS` mid-run (so even in-progress runs are protected).
- Already-done `(game, seed)` pairs (JSON with `"done": true`) are skipped. The cell is **re-runnable** and picks up where it left off.

In [ ]:
def try_drive_backup():
    """Copy local results + logs to Drive. Remounts on disconnect. Never raises."""
    global DRIVE_OUT
    if DRIVE_OUT is None:
        return
    def _copy_all():
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        (DRIVE_OUT / "logs").mkdir(exist_ok=True)
        for f in LOCAL_OUT.iterdir():
            dst = DRIVE_OUT / f.name
            if f.is_dir():
                shutil.copytree(f, dst, dirs_exist_ok=True)   # per-level result.json subdirs
            else:
                shutil.copy2(f, dst)
        for f in LOG_DIR.iterdir():
            if f.is_file():
                shutil.copy2(f, DRIVE_OUT / "logs" / f.name)
    try:
        _copy_all()
        print(f"  [Drive ✓] synced @ {datetime.now():%H:%M:%S}", flush=True)
    except OSError:
        # FUSE mount died — remount and retry once
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=True)
            DRIVE_OUT = Path("/content/drive/MyDrive/goose_results")
            _copy_all()
            print(f"  [Drive ↺] remounted + synced @ {datetime.now():%H:%M:%S}", flush=True)
        except Exception as e:
            print(f"  [Drive ✗] unavailable ({e}) — local copy safe", flush=True)


def launch(g, s):
    """Start one goose run as its own process; returns a small job dict."""
    log_path = LOG_DIR / f"log_{g}_seed{s}.txt"            # LOCAL — never hits Drive
    cmd = [sys.executable, RUNNER,
           "--game", g, "--seed", str(s),
           "--out", str(LOCAL_OUT),                        # LOCAL — never hits Drive
           "--max-actions-per-level", str(PER_LEVEL_BUDGET[g]),
           "--max-actions", str(MAX_ACTIONS[g]),
           "--max-minutes", str(MAX_MINUTES),
           "--torch-threads", str(TORCH_THREADS)]
    env = {**os.environ, "OMP_NUM_THREADS": str(TORCH_THREADS), "MKL_NUM_THREADS": str(TORCH_THREADS)}
    lf = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env)
    return {"g": g, "s": s, "proc": proc, "log": lf, "t0": time.time()}


# ─────────────────────────────────────────────────────────────────────────────
# Run ALL jobs — no skip-done check. Each run overwrites its own result files, so re-running
# a (game, seed) is always safe and gives a fresh calibration under the current config.
pending = [(g, s) for g in GAMES for s in SEEDS]
summary = []

print(f"=== Goose sweep START {datetime.now():%Y-%m-%d %H:%M:%S} ===")
print(f"    {len(pending)} jobs to run, up to {MAX_PARALLEL} in parallel on one GPU\n")

running, t_sweep, last_backup = [], time.time(), time.time()
while pending or running:
    # Top up the pool
    while pending and len(running) < MAX_PARALLEL:
        g, s = pending.pop(0)
        running.append(launch(g, s))
        print(f"  ▶ launched {g} seed{s}   ({len(running)} running, {len(pending)} queued) "
              f"@ {datetime.now():%H:%M:%S}", flush=True)

    time.sleep(5)

    # Reap finished jobs
    still = []
    for j in running:
        rc = j["proc"].poll()
        if rc is None:
            still.append(j)
            continue
        j["log"].close()
        el = (time.time() - j["t0"]) / 60
        status = "ok" if rc == 0 else f"fail(rc={rc})"
        print(f"  ✔ {j['g']} seed{j['s']}: {status}   ({el:.1f} min)", flush=True)
        summary.append({"game": j["g"], "seed": j["s"], "status": status, "elapsed_min": round(el, 1)})
        try_drive_backup()                       # back up immediately after each job
    running = still

    # Periodic backup of in-progress (incremental) results too
    if time.time() - last_backup > BACKUP_SECS:
        try_drive_backup()
        last_backup = time.time()

# ── final summary ─────────────────────────────────────────────────────────────
total_h = (time.time() - t_sweep) / 3600
print(f"\n=== Goose sweep DONE {datetime.now():%Y-%m-%d %H:%M:%S}  ({total_h:.2f} h total) ===")
for r in summary:
    extra = f"  ({r['elapsed_min']} min)" if "elapsed_min" in r else ""
    print(f"  {r['game']} seed{r['seed']}: {r['status']}{extra}")

json.dump(summary, open(LOCAL_OUT / "sweep_summary.json", "w"), indent=2)
try_drive_backup()
print(f"\nAll results in {LOCAL_OUT}")

## 4. Aggregate — steps to clear each level

Safe to run at any time (reads from local, falls back to Drive).

> The runner also writes **staircase-ready** `goose_<game>_seed<seed>_L<L>/result.json` dirs next to these summaries. To put goose on the exp_014 headline figure, copy those `goose_*_L*/` dirs into `JEPA/.../exp_014_figures_and_results/data/goose/runs/` and run `staircase.py` — no conversion step.

In [ ]:
import glob, json
import numpy as np
from collections import defaultdict

# Read from local first; if not available try Drive
_src = str(LOCAL_OUT) if LOCAL_OUT.exists() else (str(DRIVE_OUT) if DRIVE_OUT else "/content")
rows = [json.load(open(f)) for f in glob.glob(f"{_src}/goose_*.json")]

agg = defaultdict(lambda: defaultdict(list))
for r in rows:
    for L, st in r.get("level_steps", {}).items():
        agg[r["game"]][int(L)].append(st)

print(f"{'game':>5} | {'L1':>10} {'L2':>10} {'L3':>10}   [n runs / done]")
print(f"{'':->5}-+-{'':-<10}-{'':-<10}-{'':-<10}")
GAMES_AGG = ["ls20", "tu93", "re86", "g50t"]
for g in GAMES_AGG:
    cells = []
    for L in (1, 2, 3):
        v = agg[g].get(L, [])
        cells.append(f"{np.median(v):,.0f}" if v else "—")
    nr    = sum(1 for r in rows if r["game"] == g)
    ndone = sum(1 for r in rows if r["game"] == g and r.get("done"))
    print(f"{g:>5} | {cells[0]:>10} {cells[1]:>10} {cells[2]:>10}   [{ndone}/{nr}]")

print(f"\n{len(rows)} result files from {_src}")
print("(Compare to exp_014_figures_and_results staircase.)")